# 02 - Text Preprocessing

Here we turn the raw captions into a clean form the model can learn from, and we
build the **tokenizer**.

Steps:
1. Group the 5 captions under each image id (a `mapping` dictionary).
2. Clean each caption (lowercase, keep letters only, collapse spaces).
3. Wrap every caption with `startseq` ... `endseq` markers.
4. Fit a Keras `Tokenizer` on all captions.
5. Save `tokenizer.pkl`, `mapping.pkl`, and `all_captions.pkl` to `../artifacts/`.

These artifacts are used by every later notebook.

In [1]:
import os
import re
import pickle

import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer

In [2]:
BASE_DIR = "../artifacts/Flickr8k"
ARTIFACTS_DIR = "../artifacts"

In [3]:
captions_df = pd.read_csv(os.path.join(BASE_DIR, "captions.txt"))

In [4]:
captions_df.head()

,image,caption
0,1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a woo...


## 1. Build the image -> captions mapping

In [5]:
# mapping[image_id] = list of its captions
mapping = {}

for _, row in captions_df.iterrows():
    image_id = row["image"].split(".")[0]   # drop the .jpg extension
    caption = row["caption"]

    if image_id not in mapping:
        mapping[image_id] = []
    mapping[image_id].append(caption)

In [6]:
print("Images in mapping:", len(mapping))
mapping["1000268201_693b08cb0e"]

Images in mapping: 8091


['A child in a pink dress is climbing up a set of stairs in an entry way .',
 'A girl going into a wooden building .',
 'A little girl climbing into a wooden playhouse .',
 'A little girl climbing the stairs to her playhouse .',
 'A little girl in a pink dress going into a wooden cabin .']

In [7]:
def clean_caption(caption):
    caption = caption.lower()                      # lowercase
    caption = re.sub(r"[^a-zA-Z\s]", "", caption)  # keep letters and spaces only
    caption = " ".join(caption.split())            # remove extra whitespace
    caption = "startseq " + caption + " endseq"    # add start / end markers
    return caption

In [ ]:
for image_id, captions in mapping.items():
    for i in range(len(captions)):
        captions[i] = clean_caption(captions[i])

['startseq a child in a pink dress is climbing up a set of stairs in an entry way endseq',
 'startseq a girl going into a wooden building endseq',
 'startseq a little girl climbing into a wooden playhouse endseq',
 'startseq a little girl climbing the stairs to her playhouse endseq',
 'startseq a little girl in a pink dress going into a wooden cabin endseq']

In [9]:
mapping["1000268201_693b08cb0e"]

['startseq a child in a pink dress is climbing up a set of stairs in an entry way endseq',
 'startseq a girl going into a wooden building endseq',
 'startseq a little girl climbing into a wooden playhouse endseq',
 'startseq a little girl climbing the stairs to her playhouse endseq',
 'startseq a little girl in a pink dress going into a wooden cabin endseq']

## 3. Collect all captions into one list

In [10]:
all_captions = [caption for captions in mapping.values() for caption in captions]

In [11]:
len(all_captions)

40455

In [12]:
all_captions[:3]

['startseq a child in a pink dress is climbing up a set of stairs in an entry way endseq',
 'startseq a girl going into a wooden building endseq',
 'startseq a little girl climbing into a wooden playhouse endseq']

## 4. Fit the tokenizer

The tokenizer maps every word to an integer id. We then derive two numbers used
everywhere later:
- `vocab_size` = number of unique words + 1 (index 0 is reserved for padding),
- `max_length` = the length of the longest caption.

In [13]:
tokenizer = Tokenizer()

In [14]:
tokenizer.fit_on_texts(all_captions)

In [15]:
tokenizer.word_index

{'a': 1,
 'startseq': 2,
 'endseq': 3,
 'in': 4,
 'the': 5,
 'on': 6,
 'is': 7,
 'and': 8,
 'dog': 9,
 'with': 10,
 'man': 11,
 'of': 12,
 'two': 13,
 'white': 14,
 'black': 15,
 'boy': 16,
 'are': 17,
 'woman': 18,
 'girl': 19,
 'to': 20,
 'wearing': 21,
 'at': 22,
 'people': 23,
 'water': 24,
 'red': 25,
 'young': 26,
 'brown': 27,
 'an': 28,
 'his': 29,
 'blue': 30,
 'dogs': 31,
 'running': 32,
 'through': 33,
 'playing': 34,
 'while': 35,
 'down': 36,
 'shirt': 37,
 'standing': 38,
 'ball': 39,
 'little': 40,
 'grass': 41,
 'child': 42,
 'person': 43,
 'snow': 44,
 'jumping': 45,
 'over': 46,
 'front': 47,
 'three': 48,
 'sitting': 49,
 'holding': 50,
 'field': 51,
 'small': 52,
 'up': 53,
 'by': 54,
 'large': 55,
 'green': 56,
 'group': 57,
 'one': 58,
 'yellow': 59,
 'her': 60,
 'walking': 61,
 'children': 62,
 'men': 63,
 'into': 64,
 'air': 65,
 'beach': 66,
 'near': 67,
 'mouth': 68,
 'jumps': 69,
 'another': 70,
 'for': 71,
 'street': 72,
 'runs': 73,
 'its': 74,
 'from': 75,

In [16]:
vocab_size = len(tokenizer.word_index) + 1

In [17]:
vocab_size

8781

In [18]:
max_length = max(len(caption.split()) for caption in all_captions)

In [19]:
max_length

37

In [20]:
# total unique words : 8781
# max caption length : 37

In [21]:
with open(os.path.join(ARTIFACTS_DIR, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)
with open(os.path.join(ARTIFACTS_DIR, "mapping.pkl"), "wb") as f:
    pickle.dump(mapping, f)
with open(os.path.join(ARTIFACTS_DIR, "all_captions.pkl"), "wb") as f:
    pickle.dump(all_captions, f)